# PointerDoc 학습 — Colab T4

**Model**: DINOv2-small (FROZEN) + KoCharELECTRA-small (FROZEN) + sentence queries + pointer decoder.

## 업로드 파일 (colab_upload_v2/)
1. `pointerdoc_model.py`
2. `pointerdoc_train.jsonl` (~2,039 PDFs, 30K sentences)
3. `pointerdoc_images.zip` (~115MB)

In [ ]:
!pip install -q transformers==4.46.0 scipy pillow
import sys, os, json, zipfile, time, random
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    print(f'  uploaded: {name} ({len(uploaded[name])/1024:.0f}KB)')

In [ ]:
if Path('pointerdoc_images.zip').exists():
    with zipfile.ZipFile('pointerdoc_images.zip') as z:
        z.extractall('.')
    n_imgs = len(list(Path('page_images').glob('*.png')))
    print(f'Extracted {n_imgs} images')

sys.path.insert(0, '.')
from pointerdoc_model import PointerDoc, PointerDocConfig, pointerdoc_loss, hungarian_match

In [ ]:
records = []
with open('pointerdoc_train.jsonl', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

for r in records:
    name = r['image_path'].replace('\\', '/').rsplit('/', 1)[-1]
    r['image_path'] = 'page_images/' + name

print(f'Total records: {len(records)}')
print(f'Total sentences: {sum(r["n_sentences"] for r in records)}')

In [ ]:
random.seed(42)
random.shuffle(records)
n_val = max(50, len(records) // 10)
val_records = records[:n_val]
train_records = records[n_val:]
print(f'Train: {len(train_records)}, Val: {len(val_records)}')

In [ ]:
from transformers import AutoImageProcessor, AutoTokenizer
VISION_ID = 'facebook/dinov2-small'   # without-registers (transformers 4.46 호환)
TEXT_ID = 'monologg/kocharelectra-small-discriminator'
processor = AutoImageProcessor.from_pretrained(VISION_ID)
tokenizer = AutoTokenizer.from_pretrained(TEXT_ID)
MAX_ROWS = 128
MAX_TOKENS = 48

class PointerDocDataset(Dataset):
    def __init__(self, records, processor, tokenizer, max_rows=128, max_tokens=48):
        self.records = records
        self.processor = processor
        self.tokenizer = tokenizer
        self.max_rows = max_rows
        self.max_tokens = max_tokens
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        img = Image.open(r['image_path']).convert('RGB')
        pixel = self.processor(images=img, return_tensors='pt')['pixel_values'][0]

        n = min(len(r['bboxes']), self.max_rows)
        bboxes = torch.tensor(r['bboxes'][:n], dtype=torch.float32)

        row_mask = torch.zeros(self.max_rows, dtype=torch.bool)
        row_mask[:n] = True
        padded_bboxes = torch.zeros(self.max_rows, 4, dtype=torch.float32)
        padded_bboxes[:n] = bboxes

        texts = r['row_texts'][:n]
        if texts:
            tok = self.tokenizer(texts, padding='max_length', truncation=True,
                                 max_length=self.max_tokens, return_tensors='pt')
            ti = tok['input_ids']
            tm = tok['attention_mask']
        else:
            ti = torch.zeros(0, self.max_tokens, dtype=torch.long)
            tm = torch.zeros(0, self.max_tokens, dtype=torch.long)
        padded_text_ids = torch.zeros(self.max_rows, self.max_tokens, dtype=torch.long)
        padded_text_mask = torch.zeros(self.max_rows, self.max_tokens, dtype=torch.long)
        if n > 0:
            padded_text_ids[:n] = ti
            padded_text_mask[:n] = tm

        sent_ids = [[i for i in s if i < n] for s in r['sentence_row_ids']]
        sent_ids = [s for s in sent_ids if s]

        return {
            'pixel_values': pixel,
            'bboxes': padded_bboxes,
            'row_mask': row_mask,
            'text_ids': padded_text_ids,
            'text_mask': padded_text_mask,
            'sentence_row_ids': sent_ids,
            'pdf': r['pdf'],
            'row_texts': r['row_texts'][:n],
            'sentence_texts': r['sentence_texts'],
        }

def collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'bboxes': torch.stack([b['bboxes'] for b in batch]),
        'row_mask': torch.stack([b['row_mask'] for b in batch]),
        'text_ids': torch.stack([b['text_ids'] for b in batch]),
        'text_mask': torch.stack([b['text_mask'] for b in batch]),
        'sentence_row_ids': [b['sentence_row_ids'] for b in batch],
        'pdfs': [b['pdf'] for b in batch],
        'row_texts': [b['row_texts'] for b in batch],
        'sentence_texts': [b['sentence_texts'] for b in batch],
    }

train_ds = PointerDocDataset(train_records, processor, tokenizer, MAX_ROWS, MAX_TOKENS)
val_ds = PointerDocDataset(val_records, processor, tokenizer, MAX_ROWS, MAX_TOKENS)
BATCH = 4
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}')

In [ ]:
cfg = PointerDocConfig(
    max_rows=MAX_ROWS, num_queries=32, d_model=256, num_decoder_layers=4,
    text_max_length=MAX_TOKENS, use_text=True, use_vision=True,
    vision_model_id='facebook/dinov2-small',
)
model = PointerDoc(cfg).to(device)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'Total params: {n_total/1e6:.1f}M, Trainable: {n_trainable/1e6:.1f}M')

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
EPOCHS = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
best_val = float('inf')
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    t0 = time.time()
    for step, batch in enumerate(train_loader):
        pixel = batch['pixel_values'].to(device)
        bboxes = batch['bboxes'].to(device)
        mask = batch['row_mask'].to(device)
        text_ids = batch['text_ids'].to(device)
        text_mask = batch['text_mask'].to(device)
        out = model(pixel, bboxes, mask, text_ids=text_ids, text_mask=text_mask)
        loss_dict = pointerdoc_loss(out, batch['sentence_row_ids'], mask)
        loss = loss_dict['loss']
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            pixel = batch['pixel_values'].to(device)
            bboxes = batch['bboxes'].to(device)
            mask = batch['row_mask'].to(device)
            text_ids = batch['text_ids'].to(device)
            text_mask = batch['text_mask'].to(device)
            out = model(pixel, bboxes, mask, text_ids=text_ids, text_mask=text_mask)
            loss_dict = pointerdoc_loss(out, batch['sentence_row_ids'], mask)
            val_loss += loss_dict['loss'].item()
    train_loss /= max(len(train_loader), 1)
    val_loss /= max(len(val_loader), 1)
    elapsed = time.time() - t0
    print(f'Epoch {epoch+1:2d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  ({elapsed:.0f}s)')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'state_dict': model.state_dict(), 'cfg': cfg.__dict__}, 'pointerdoc_best.pt')
        print('  > saved best')

print(f'Best val: {best_val:.4f}')

In [ ]:
from google.colab import files
files.download('pointerdoc_best.pt')